# 01 · Limpeza dos dados

**Projeto Final · Visualização de Dados · ** — Etapa 2: Preparação dos dados

Este notebook prepara os dois datasets do projeto e responde às perguntas da Etapa 1 só com números:

Cada bloco de código vem depois de uma célula de texto que explica **o que** foi feito e **por quê**.

### Perguntas da Etapa 1

**Músicas (Spotify)**
1. Quais gêneros de playlist têm as músicas mais populares?
2. A popularidade das músicas muda conforme o ano de lançamento?
3. Características do áudio (energia, acusticidade, fala, ao vivo) têm relação com a popularidade?
4. As músicas ficaram mais curtas ao longo das décadas?

**Filmes (TMDB)**

5. Quais gêneros de filme têm as maiores notas médias?
6. Como a quantidade de filmes e a nota média mudam por década de lançamento?
7. Filmes com mais votos também têm notas mais altas?
8. Quais idiomas originais aparecem mais e como são avaliados?

In [2]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")
from src.utils import DATA, LIMPOS, GENEROS_MUSICA, IDIOMAS

pd.set_option("display.max_columns", 30)
LIMPOS.mkdir(exist_ok=True)

Erro: Nenhuma conexão selecionada.

---
## Parte A · Músicas (Spotify)

### A1. Carregar
O dataset `spotify_songs.csv` lista músicas de playlists do Spotify, com popularidade (0 a 100), gênero da playlist e características de áudio calculadas pelo Spotify.

In [3]:
musicas = pd.read_csv(DATA / "musica" / "spotify_songs.csv")

Erro: Nenhuma conexão selecionada.

### A2. Inspecionar
Antes de mexer, conferimos o tamanho, os tipos e os valores ausentes.

In [4]:
print("Linhas x colunas:", musicas.shape)
musicas.info()

Erro: Nenhuma conexão selecionada.

In [5]:
musicas.head()

Erro: Nenhuma conexão selecionada.

In [6]:
musicas.isna().sum()[lambda s: s > 0]

Erro: Nenhuma conexão selecionada.

**O que a inspeção mostrou:**
- 32.833 linhas e 23 colunas.
- A data de lançamento do álbum (`track_album_release_date`) está como **texto** e com formatos diferentes: `2019-06-14`, `2019-06` ou só `2019`.
- A duração está em milissegundos, pouco legível.
- 5 músicas não têm nome, artista nem álbum.
- Uma mesma música (`track_id`) aparece em várias playlists, o que é esperado. Mas também há músicas repetidas **dentro da mesma playlist**, o que é duplicação de verdade (tratada no passo A6).

### A3. Selecionar e renomear
Ficamos só com as colunas que ajudam a responder às perguntas 1 a 4 e traduzimos os nomes para português (seguindo o tratamento que a equipe já tinha começado em `songs_tratamento.ipynb`).

Mantivemos `track_id` e `playlist_id` por enquanto, porque eles são necessários para encontrar duplicados. `playlist_id` é descartado depois do passo A6.

Removemos `danceability`, `key`, `loudness`, `mode`, `instrumentalness`, `valence` e `tempo`, que não entram nas perguntas.

In [7]:
colunas_descartadas = ["track_album_id", "danceability", "key", "loudness", "mode",
                       "instrumentalness", "valence", "tempo"]
musicas = musicas.drop(columns=colunas_descartadas)

musicas = musicas.rename(columns={
    "track_id": "id_musica",
    "track_name": "nome_musica",
    "track_artist": "artista",
    "track_popularity": "popularidade_musica",
    "track_album_name": "album_musica",
    "track_album_release_date": "data_lancamento_album",
    "playlist_name": "nome_playlist",
    "playlist_id": "id_playlist",
    "playlist_genre": "genero_playlist",
    "playlist_subgenre": "subgenero_playlist",
    "energy": "energia",
    "speechiness": "fala",
    "acousticness": "acusticidade",
    "liveness": "prob_aovivo",
    "duration_ms": "duracao_ms",
})
musicas.columns.tolist()

Erro: Nenhuma conexão selecionada.

### A4. Corrigir tipos
- **Ano de lançamento:** a data vem em três formatos. O ano (4 primeiros caracteres) existe em todos, então extraímos `ano_lancamento` como inteiro e criamos `decada` para a pergunta 4. Não usamos a data completa porque 1.886 linhas só têm o ano.
- **Duração:** convertemos de milissegundos para minutos (`duracao_min`) e **removemos `duracao_ms`**. No notebook anterior, o `drop` não tinha sido atribuído de volta ao DataFrame, então a coluna continuava lá.

In [8]:
print(musicas["data_lancamento_album"].str.len().value_counts())

musicas["ano_lancamento"] = musicas["data_lancamento_album"].str[:4].astype(int)
musicas["decada"] = (musicas["ano_lancamento"] // 10) * 10

musicas["duracao_min"] = (musicas["duracao_ms"] / 60000).round(2)
musicas = musicas.drop(columns=["duracao_ms", "data_lancamento_album"])

musicas[["ano_lancamento", "decada", "duracao_min"]].describe()

Erro: Nenhuma conexão selecionada.

### A5. Tratar ausentes
Apenas 5 linhas (0,02%) não têm nome, artista e álbum. Sem o nome não dá para identificar a música, e a perda é desprezível, então **removemos** essas linhas.

In [9]:
print("Antes:", len(musicas))
musicas = musicas.dropna(subset=["nome_musica", "artista"])
print("Depois:", len(musicas))
musicas.isna().sum().sum()

Erro: Nenhuma conexão selecionada.

### A6. Remover duplicados
Há dois tipos de repetição, e tratamos cada um de forma diferente:

1. **A mesma música repetida na mesma playlist** (mesmo `id_musica` e `id_playlist`). Isso é erro de coleta e inflaria contagens e médias, então **removemos**.
2. **A mesma música em playlists diferentes.** Isso é informação real (a música está em várias playlists, às vezes de gêneros diferentes), então **mantemos**. Cada linha passa a significar "uma música dentro de uma playlist".

Para as perguntas que são sobre a música em si (2, 3 e 4), usamos a versão **sem repetição de `id_musica`** no passo de estatísticas, para não contar a mesma música duas vezes.

In [10]:
print("Duplicados exatos:", musicas.duplicated().sum())
print("Mesma música na mesma playlist:", musicas.duplicated(subset=["id_musica", "id_playlist"]).sum())

musicas = musicas.drop_duplicates(subset=["id_musica", "id_playlist"])
musicas = musicas.drop(columns=["id_playlist"])

print("Linhas após remover:", len(musicas))
print("Músicas únicas:", musicas["id_musica"].nunique())

Erro: Nenhuma conexão selecionada.

### A7. Padronizar
- Removemos espaços extras dos textos.
- Trocamos os códigos de gênero (`r&b`, `latin`...) por nomes de exibição (`R&B`, `Latina`...), que vão direto para os gráficos.
- **Durações muito curtas:** há faixas com poucos segundos (vinhetas e intros). Removemos as que têm menos de 1 minuto, porque não são músicas completas e distorcem a pergunta 4.

In [11]:
for col in ["nome_musica", "artista", "album_musica", "nome_playlist", "subgenero_playlist"]:
    musicas[col] = musicas[col].str.strip()

musicas["genero_playlist"] = musicas["genero_playlist"].map(GENEROS_MUSICA)
print("Gêneros sem correspondência:", musicas["genero_playlist"].isna().sum())

curtas = musicas["duracao_min"] < 1
print("Faixas com menos de 1 minuto:", curtas.sum())
musicas = musicas[~curtas]
print("Linhas finais:", len(musicas))

Erro: Nenhuma conexão selecionada.

### A8. Salvar

In [12]:
musicas.to_csv(LIMPOS / "musicas_limpas.csv", index=False)
musicas.head()

Erro: Nenhuma conexão selecionada.

---
## Parte B · Filmes (TMDB)

### B1. Carregar
O `Top Movies dataset.csv` traz quase 10 mil filmes populares do TMDB, com data de lançamento, popularidade, número de votos, nota média, idioma original e gêneros.

In [13]:
filmes = pd.read_csv(DATA / "filmes" / "top_movies_dataset.csv")

Erro: Nenhuma conexão selecionada.

### B2. Inspecionar

In [14]:
print("Linhas x colunas:", filmes.shape)
filmes.info()

Erro: Nenhuma conexão selecionada.

In [15]:
filmes.head()

Erro: Nenhuma conexão selecionada.

In [16]:
filmes.isna().sum()

Erro: Nenhuma conexão selecionada.

In [17]:
# Linhas com problema: sem popularidade
filmes[filmes["Popularity"].isna()][["Release_Date", "Title", "Overview", "Working URL"]]

Erro: Nenhuma conexão selecionada.

**O que a inspeção mostrou:**
- **Números guardados como texto:** `Vote_Count` e `Vote_Average` estão como `str`.
- **Data como texto** no formato brasileiro `dd/mm/aaaa`.
- **11 linhas quebradas.** A sinopse de *Pixie Hollow Bake Off* tem quebras de linha no CSV original, e cada pedaço virou uma "linha" nova (`- Just Desserts`, `- Dust Up`...). Essas linhas não têm nenhum dado numérico e vêm com `Working URL = 0`.
- `Genre` junta vários gêneros num texto só (`"Action, Adventure, Science Fiction"`).
- `Poster_Url`, `Working URL` e `Overview` não servem para as perguntas.

### B3. Selecionar e renomear
Descartamos as colunas de imagem e a sinopse, e traduzimos os nomes.

In [18]:
filmes = filmes.drop(columns=["Overview", "Poster_Url", "Working URL"])
filmes = filmes.rename(columns={
    "Release_Date": "data_lancamento",
    "Title": "titulo",
    "Popularity": "popularidade",
    "Vote_Count": "qtd_votos",
    "Vote_Average": "nota_media",
    "Original_Language": "idioma_original",
    "Genre": "generos",
})
filmes.columns.tolist()

Erro: Nenhuma conexão selecionada.

### B4. Corrigir tipos
- `qtd_votos` e `nota_media` viram números com `pd.to_numeric(errors="coerce")`. O que não for número vira `NaN` e é tratado no próximo passo.
- `data_lancamento` vira data com o formato explícito `%d/%m/%Y`. Sem o formato, o pandas poderia trocar dia e mês.
- Criamos `ano` e `decada` a partir da data.

In [19]:
filmes["qtd_votos"] = pd.to_numeric(filmes["qtd_votos"], errors="coerce")
filmes["nota_media"] = pd.to_numeric(filmes["nota_media"], errors="coerce")
filmes["data_lancamento"] = pd.to_datetime(filmes["data_lancamento"], format="%d/%m/%Y", errors="coerce")

filmes.dtypes

Erro: Nenhuma conexão selecionada.

### B5. Tratar ausentes
1. **Linhas quebradas:** as 11 linhas sem popularidade são os pedaços da sinopse quebrada (e o filme a que ela pertence, que ficou sem nenhum dado numérico). Não há como recuperar os números, então **removemos**.
2. **Filmes com 0 votos têm nota 0.** Isso não é uma nota ruim: é **ausência de nota** (normalmente filmes ainda não lançados). Trocamos a nota desses filmes por `NaN` em vez de apagar o filme, para que ele continue valendo nas contagens por ano e idioma.
3. **Notas pouco confiáveis:** uma nota calculada com 3 votos não diz muito. Criamos a coluna `nota_confiavel` (verdadeiro quando há **50 votos ou mais**) e usamos só esses filmes nas perguntas sobre nota.

In [20]:
print("Antes:", len(filmes))
filmes = filmes.dropna(subset=["popularidade", "qtd_votos", "data_lancamento"])
print("Depois de remover linhas quebradas:", len(filmes))

sem_voto = filmes["qtd_votos"] == 0
print("Filmes com 0 votos (nota vira NaN):", sem_voto.sum())
filmes.loc[sem_voto, "nota_media"] = np.nan

filmes["qtd_votos"] = filmes["qtd_votos"].astype(int)
filmes["ano"] = filmes["data_lancamento"].dt.year
filmes["decada"] = (filmes["ano"] // 10) * 10
filmes["nota_confiavel"] = filmes["qtd_votos"] >= 50
print("Filmes com nota confiável (50+ votos):", filmes["nota_confiavel"].sum())

filmes.isna().sum()

Erro: Nenhuma conexão selecionada.

### B6. Remover duplicados
Não há linhas exatamente iguais. Há **títulos repetidos**, mas ao conferir eles são **remakes** com datas e números diferentes (ex.: *A Nightmare on Elm Street* de 1984 e de 2010). Por isso, o critério de duplicado é **título + data de lançamento**, e não só o título.

In [21]:
print("Duplicados exatos:", filmes.duplicated().sum())
print("Títulos repetidos:", filmes.duplicated(subset=["titulo"]).sum())
print("Título + data repetidos:", filmes.duplicated(subset=["titulo", "data_lancamento"]).sum())

filmes = filmes.drop_duplicates(subset=["titulo", "data_lancamento"])
filmes[filmes.duplicated(subset=["titulo"], keep=False)].sort_values("titulo")[["titulo", "ano", "qtd_votos", "nota_media"]].head(6)

Erro: Nenhuma conexão selecionada.

### B7. Padronizar
- Removemos espaços extras do começo e do fim dos títulos.
- Criamos `idioma` com o nome em português dos códigos de idioma (`en` → Inglês). Os códigos sem tradução no dicionário ficam como `Outros`.
- **Gêneros:** um filme pode ter vários. Para a pergunta 5, criamos uma tabela separada (`filmes_generos.csv`) com **uma linha por filme e gênero**, usando `str.split` e `explode`. Um filme de "Ação, Aventura" conta nos dois gêneros. Também guardamos o `genero_principal` (o primeiro da lista).

In [22]:
filmes["titulo"] = filmes["titulo"].str.strip()
filmes["idioma"] = filmes["idioma_original"].map(IDIOMAS).fillna("Outros")

filmes["genero_principal"] = filmes["generos"].str.split(",").str[0].str.strip()

filmes_generos = (
    filmes.assign(genero=filmes["generos"].str.split(","))
          .explode("genero")
          .assign(genero=lambda d: d["genero"].str.strip())
          [["titulo", "ano", "decada", "genero", "qtd_votos", "nota_media", "nota_confiavel", "popularidade"]]
)
print("Filmes:", len(filmes), "| linhas filme-gênero:", len(filmes_generos))
filmes_generos["genero"].value_counts()

Erro: Nenhuma conexão selecionada.

### B8. Salvar

In [23]:
filmes.to_csv(LIMPOS / "filmes_limpos.csv", index=False)
filmes_generos.to_csv(LIMPOS / "filmes_generos.csv", index=False)
filmes.head()

Erro: Nenhuma conexão selecionada.